# CTR / Engagement Opportunity Scoring: notebook 2

Notebook 1 found two things that need checking before any ranking can be trusted:

1. The model barely beat the baseline (R2 about 0.05), so "expected CTR" is a weak yardstick.
2. The top of the ranking is pages with very large impressions at good positions and a tiny CTR. That could be real page-level opportunity, or it could be a *client-level* effect (some clients simply have low CTR everywhere), which would make whole clients look like "underperformers".

This notebook measures how much of CTR is a client effect, tests a better-suited model (clicks as counts, weighted by impressions), and compares a global score with a client-relative score. Everything is *observed / directional*.

In [ ]:
# 1. Setup, load, same analysis set as notebook 1. Needs Colab secret HF_TOKEN.
!pip install -q datasets huggingface_hub
from google.colab import userdata
from huggingface_hub import login
login(userdata.get("HF_TOKEN"))

import numpy as np
import pandas as pd
from datasets import load_dataset

ds = load_dataset("FlyRank/internship-lanes", "engagement_fix")
df = ds["train"].to_pandas()

MIN_IMP = 100
d = df[(df["impressions_30d"] >= MIN_IMP) & df["avg_position_30d"].notna()].copy()
d["ctr"] = d["clicks_30d"] / d["impressions_30d"]
d["log_impressions"] = np.log1p(d["impressions_30d"])
CAT = ["content_type", "main_intent", "age_tier", "freshness_tier", "word_count_tier", "char_count_tier"]
for c in CAT:
    d[c] = d[c].astype(str)
print(f"{len(d):,} pages, {d['client_hash_id'].nunique()} clients")
print("needs_engagement_fix is True for", f"{df['needs_engagement_fix'].mean():.1%}", "of pages (a flag that is always True cannot rank anything)")

## How much of CTR is just "which client is it"?

In [ ]:
# 2. Client effect: how different are clients, and how much of CTR variation do they explain?
per_client = d.groupby("client_hash_id").agg(
    pages=("ctr", "size"),
    impressions=("impressions_30d", "sum"),
    clicks=("clicks_30d", "sum"),
    median_pos=("avg_position_30d", "median"),
)
per_client["pooled_ctr"] = per_client["clicks"] / per_client["impressions"]
print("Pooled CTR across clients (min / median / max):",
      f"{per_client['pooled_ctr'].min():.4f} / {per_client['pooled_ctr'].median():.4f} / {per_client['pooled_ctr'].max():.4f}")
print("Biggest client share of all impressions:", f"{per_client['impressions'].max() / per_client['impressions'].sum():.1%}")
print("Biggest client share of all pages     :", f"{per_client['pages'].max() / per_client['pages'].sum():.1%}")

# share of variation in log CTR explained by client identity (clicks > 0 only, to take logs)
pos = d[d["clicks_30d"] > 0].copy()
pos["lctr"] = np.log(pos["ctr"])
between = pos.groupby("client_hash_id")["lctr"].transform("mean")
r2_client = 1 - ((pos["lctr"] - between) ** 2).sum() / ((pos["lctr"] - pos["lctr"].mean()) ** 2).sum()
print(f"Share of variation in log CTR explained by client alone: {r2_client:.1%} (pages with >=1 click: {len(pos):,})")
print("Pages with zero clicks:", f"{(d['clicks_30d'] == 0).mean():.1%}")

## A model better suited to the data

CTR is clicks out of impressions. A page with 500,000 impressions tells us far more than one with 100. So: predict clicks per impression with a **Poisson** loss and **weight each page by its impressions**, and judge it with impression-weighted error. The baseline becomes the pooled CTR per position bucket (total clicks / total impressions), which is the fair like-for-like version.

In [ ]:
# 3. Weighted baseline vs Poisson model, held-out clients
from sklearn.model_selection import GroupShuffleSplit, GroupKFold
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, r2_score

NUM = ["avg_position_30d", "log_impressions"]
X, y, w, g = d[CAT + NUM], d["ctr"], d["impressions_30d"], d["client_hash_id"]
tr, te = next(GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42).split(X, y, groups=g))
assert set(g.iloc[tr]).isdisjoint(set(g.iloc[te])), "client leakage!"

bins = [0, 1.5, 2.5, 3.5, 4.5, 5.5, 7.5, 10.5, 15.5, 20.5, 1000]
def bucket(s): return pd.cut(s, bins=bins, labels=False)
tr_df = d.iloc[tr]
grp = tr_df.groupby(bucket(tr_df["avg_position_30d"]))
pooled = grp["clicks_30d"].sum() / grp["impressions_30d"].sum()
te_df = d.iloc[te]
base_pred = bucket(te_df["avg_position_30d"]).map(pooled).fillna(tr_df["clicks_30d"].sum() / tr_df["impressions_30d"].sum())

def make_model():
    pre = ColumnTransformer([("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), CAT)],
                            remainder="passthrough")
    return Pipeline([("pre", pre), ("gbm", HistGradientBoostingRegressor(
        loss="poisson", max_iter=200, learning_rate=0.06, random_state=42))])

model = make_model()
model.fit(X.iloc[tr], y.iloc[tr], gbm__sample_weight=w.iloc[tr])
pred = model.predict(X.iloc[te])

yt, wt = y.iloc[te], w.iloc[te]
res = {}
for name, p in [("Baseline (pooled CTR by position)", base_pred), ("Poisson model", pred)]:
    res[name] = (mean_absolute_error(yt, p, sample_weight=wt), r2_score(yt, p, sample_weight=wt))
    print(f"{name}: impression-weighted MAE {res[name][0]:.5f} | weighted R2 {res[name][1]:.3f}")
b, m = res["Baseline (pooled CTR by position)"], res["Poisson model"]
print(f"Weighted MAE change vs baseline: {(m[0] - b[0]) / b[0]:+.1%}")

## Two scores, compared

* **Global score:** expected CTR from the model, which only knows position, impressions and content features.
* **Client-relative score:** the same expectation, rescaled so each client's *total* expected clicks matches its *total* actual clicks. That removes "this whole client has low CTR" and leaves pages that fall short *compared with their own client's other pages*.

If the two rankings disagree a lot, the global ranking was mostly measuring clients, not pages.

In [ ]:
# 4. Out-of-fold expectations, then global vs client-relative missed clicks
k = min(5, g.nunique())
from sklearn.model_selection import cross_val_predict
oof = np.zeros(len(d))
for tr_i, te_i in GroupKFold(n_splits=k).split(X, y, groups=g):
    mdl = make_model()
    mdl.fit(X.iloc[tr_i], y.iloc[tr_i], gbm__sample_weight=w.iloc[tr_i])
    oof[te_i] = mdl.predict(X.iloc[te_i])
d["expected_ctr"] = np.clip(oof, 1e-6, 1)

d["exp_clicks"] = d["expected_ctr"] * d["impressions_30d"]
fac = d.groupby("client_hash_id").apply(lambda s: s["clicks_30d"].sum() / s["exp_clicks"].sum(), include_groups=False)
d["client_factor"] = d["client_hash_id"].map(fac).clip(0.05, 5)
d["rel_expected_ctr"] = np.clip(d["expected_ctr"] * d["client_factor"], 0, 1)

d["missed_global"] = ((d["expected_ctr"] - d["ctr"]).clip(lower=0) * d["impressions_30d"])
d["missed_relative"] = ((d["rel_expected_ctr"] - d["ctr"]).clip(lower=0) * d["impressions_30d"])

N = min(1000, len(d) // 10)
top_g = d.nlargest(N, "missed_global")
top_r = d.nlargest(N, "missed_relative")
overlap = len(set(top_g["content_hash_id"]) & set(top_r["content_hash_id"])) / N

def conc(t):
    s = t["client_hash_id"].value_counts()
    return t["client_hash_id"].nunique(), s.iloc[0] / len(t)

cg, cr = conc(top_g), conc(top_r)
print(f"Top {N} overlap between the two rankings: {overlap:.1%}")
print(f"Global score : top {N} come from {cg[0]} clients; largest client = {cg[1]:.1%}")
print(f"Relative score: top {N} come from {cr[0]} clients; largest client = {cr[1]:.1%}")
print("Median client_factor (1.0 = client in line with the model):", round(float(fac.median()), 3),
      "| min", round(float(fac.min()), 3), "| max", round(float(fac.max()), 3))

In [ ]:
# 5. Client-relative top pages with reason codes (local use only, do NOT publish IDs)
def reasons(r):
    out = []
    if r["rel_expected_ctr"] > r["ctr"] and r["ctr"] < 0.5 * r["rel_expected_ctr"]: out.append("CTR under half of client-adjusted expectation")
    elif r["rel_expected_ctr"] > r["ctr"]: out.append("CTR below client-adjusted expectation")
    if 3.5 < r["avg_position_30d"] <= 10.5: out.append("striking distance (position 4-10)")
    if r["impressions_30d"] >= d["impressions_30d"].quantile(0.75): out.append("high visibility")
    return "; ".join(out) if out else "no action flagged"

top_r = top_r.copy()
top_r["reason_codes"] = top_r.apply(reasons, axis=1)
show = ["content_hash_id", "avg_position_30d", "impressions_30d", "ctr", "rel_expected_ctr", "missed_relative", "reason_codes"]
print(top_r[show].head(10).round(4).to_string(index=False))
print()
print("Reason codes in client-relative top", N)
print(top_r["reason_codes"].value_counts().head(5).to_string())

import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(d["missed_global"].clip(lower=1), d["missed_relative"].clip(lower=1), s=3, alpha=0.2)
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("Missed clicks, global score"); ax.set_ylabel("Missed clicks, client-relative score")
ax.set_title("Do the two scores agree?")
plt.tight_layout(); plt.show()

In [ ]:
# 6. Numbers to copy back to me (no IDs, no client info)
print("===== COPY EVERYTHING BELOW THIS LINE AND SEND IT TO ME =====")
print(f"Pages {len(d)} | clients {d['client_hash_id'].nunique()} | zero-click pages {(d['clicks_30d'] == 0).mean():.1%}")
print(f"Pooled CTR by client min/median/max: {per_client['pooled_ctr'].min():.4f} / {per_client['pooled_ctr'].median():.4f} / {per_client['pooled_ctr'].max():.4f}")
print(f"Biggest client: {per_client['impressions'].max() / per_client['impressions'].sum():.1%} of impressions, {per_client['pages'].max() / per_client['pages'].sum():.1%} of pages")
print(f"Share of log-CTR variation explained by client: {r2_client:.1%}")
print(f"Weighted baseline: MAE {b[0]:.5f} R2 {b[1]:.3f} | Poisson model: MAE {m[0]:.5f} R2 {m[1]:.3f} | MAE change {(m[0]-b[0])/b[0]:+.1%}")
print(f"Top {N} overlap global vs relative: {overlap:.1%}")
print(f"Global top {N}: {cg[0]} clients, largest {cg[1]:.1%} | Relative top {N}: {cr[0]} clients, largest {cr[1]:.1%}")
print(f"client_factor min/median/max: {fac.min():.3f} / {fac.median():.3f} / {fac.max():.3f}")
print("Relative reason codes:")
print(top_r["reason_codes"].value_counts().head(4).to_string())